# EXP-2026-001 on Colab: text control (Test A) and VLM captions (Test B), no training

The checks were fixed in `configs/experiments/EXP-2026-001.json` (block `text_control_captions_v1`) before this run. The notebook loads the step-500 UNet saved by r3 and the same 16 training slices.

**Test A, text control.** Every generation uses the declared pre-op-pair start at t = 981; only the text changes:

| Scenario | Classes requested |
|---|---|
| `own` | each slice's own target-informed prompt (repeats r4) |
| `none` | no annotated compartment |
| `id2` | peritumoral signal alteration |
| `id2_id3` | peritumoral signal alteration + enhancing tumor |

The question is whether the generated mask's classes follow the request, including on the 32 counterfactual generations, where the request differs from the slice's own classes.

**Test B, VLM captions.** An open general VLM (Qwen2.5-VL-7B) and an open medical VLM (MedGemma-4B) describe the 16 **pre-op** slices, which are available at inference, as short terms and as sentences. Both run locally on this GPU; nothing goes to an external service. The notebook records CLIP token counts against the limit of 77 and checks a keyword flag against the pre-op labels. Nothing is trained on the captions.

MedGemma is gated. To include it, accept its terms on its Hugging Face page and add your token as a Colab secret named `HF_TOKEN` (key icon in the left bar). Without it, MedGemma is skipped and the run continues.

Training data only; about 25 minutes. Outputs go once to `synthesis/EXP-2026-001/text_control_captions/<RUN_TAG>/` on Drive. The PNG pages and arrays show patient images, so keep them private.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "exp-2026-001-implementation"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
WORK = "/content/MedSegFactory"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
RUN_TAG = "r1"  # use a new tag for every attempt; a run folder is never reused
WEIGHTS = "data/RHUH-GBM/synthesis/EXP-2026-001/stage4/r3/unet_step000500.pt"  # saved by r3

## 1. Mount Drive, get the latest code from the fork, link the data

In [ ]:
import copy
import gc
import json
import os
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN secret found")
except Exception as exc:
    print(f"No HF_TOKEN secret ({type(exc).__name__}): gated models such as MedGemma will be skipped")
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
assert (PRIVATE / "split/split.json").exists(), "The synthesis folder is missing on Drive"
assert Path(WEIGHTS).exists(), f"{WEIGHTS} is missing: r3 must have run with SAVE_UNET_TO_DRIVE = True"
RUN = PRIVATE / "text_control_captions" / RUN_TAG
assert not RUN.exists(), f"{RUN} exists: choose a new RUN_TAG (records are never overwritten)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "nibabel>=5.3,<6", "SimpleITK==2.5.6", "pydicom>=3,<4", "scikit-image>=0.25"], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)
STARTED = time.perf_counter()

## 2. Models: MedSegFactory checkpoint (strict), frozen VAE and CLIP, sampler

In [ ]:
import numpy as np
import torch
from diffusers import AutoencoderKL
from huggingface_hub import hf_hub_download
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.manifest import file_sha256
from rhuh_synthesis.model import reference_ddim_scheduler
from rhuh_synthesis.provenance import code_version, config_sha256, environment
from rhuh_synthesis.stage3 import load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))
S4 = config["stage4_overfit"]
DIAG = config["stage4_r2_diagnostics"]  # metric definitions
TP = config["text_control_captions_v1"]
checkpoint = hf_hub_download(MSF_REPO, MSF_FILE)
unet_config = UNet2DConditionModel.load_config(SD_REPO, subfolder="unet")
unet4, load_report = load_reference_unet(UNet2DConditionModel, unet_config, checkpoint)
assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"
vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
ddim = reference_ddim_scheduler()
record = {
    "stage": "text_control_captions_v1", "run_tag": RUN_TAG, "commit": commit, "code_version": code_version(WORK),
    "config_sha256": config_sha256(config), "text_control_captions_config": TP, "environment": environment(),
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "checkpoint": {"repo": MSF_REPO, "file": MSF_FILE, "sha256": file_sha256(checkpoint),
                   "missing_keys": len(load_report["missing_keys"]), "unexpected_keys": len(load_report["unexpected_keys"])},
}
print({k: record[k] for k in ("commit", "config_sha256", "torch", "cuda", "gpu")}, record["checkpoint"])
print("CLIP max tokens:", tokenizer.model_max_length)

## 3. Load the r3 step-500 weights (strict) into the 11-channel UNet

In [ ]:
from rhuh_synthesis.overfit import load_trained_unet

unet11, record["weights"] = load_trained_unet(unet4, WEIGHTS, device="cuda")
record["weights"]["path"] = WEIGHTS
del unet4
torch.cuda.empty_cache()
print(record["weights"])

## 4. Rebuild the 16 training slices and cache their features

In [ ]:
from rhuh_synthesis.overfit import label_volumes, ranked_patients, restrict, select_overfit_set
from rhuh_synthesis.representation import crop_forward
from rhuh_synthesis.stage3 import clip_encoder
from rhuh_synthesis.training import build_feature_cache
from rhuh_synthesis.working_grid import to_working_grid

split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
train_patients = [p for p in split["roles"]["train"]["patients"] if p not in policy["excluded_training"]]
paths = {p: next(PRIVATE.glob(f"export_v1/train*/{p}.npz")) for p in train_patients}
volumes = {}
for patient, path in paths.items():
    with np.load(path) as data:
        volumes[patient] = label_volumes(data["label"])
ranked = ranked_patients(volumes)


def load_grid(patient):
    with np.load(paths[patient]) as data:
        arrays = {key: data[key] for key in data.files}
    crop, grid = to_working_grid(arrays)
    grid["post_support"] = crop_forward(arrays["post_support"].astype(np.uint8), crop).astype(bool)
    return grid


selection, grids, skipped = select_overfit_set(ranked, load_grid, positive=S4["patients"]["positive"],
                                               negative=S4["patients"]["negative"], **S4["slices_per_patient"])
patients = [entry["patient"] for entry in selection]
exports = {entry["patient"]: restrict(grids[entry["patient"]], [z for z, _ in entry["slices"]]) for entry in selection}
items = [(entry["patient"], k) for entry in selection for k in range(len(entry["slices"]))]
categories = [category for entry in selection for _, category in entry["slices"]]
groups = [patient for patient, _ in items]
letters = {entry["patient"]: "ABCD"[n] for n, entry in enumerate(selection)}
names = [f"{letters[p]}-{c}-{k + 1}" for (p, k), c in zip(items, categories)]
targets = [{"post_image": exports[p]["post_image"][k], "pre_image": exports[p]["pre_image"][k],
            "label": exports[p]["label"][k], "region": exports[p]["pre_foreground"][k] & exports[p]["post_support"][k],
            "brain": exports[p]["pre_foreground"][k]} for p, k in items]
pixel_mm = config["representation"]["crop_mm"] / config["representation"]["out_px"]
del grids
encode = clip_encoder(tokenizer, text_encoder, "cuda")
cache = build_feature_cache(exports, vae, encode, config["text"], batch=8, device="cuda",
                            mask_codec=config["representation"]["mask_codec"])
record["selection"] = [{"letter": letters[e["patient"]], "patient": e["patient"], "kind": e["kind"],
                        "slices": [{"frame_z": z, "category": c} for z, c in e["slices"]]} for e in selection]
print("skipped (rank, kind, reason):", [(ranked[s["kind"]].index(s["patient"]) + 1, s["kind"], s["reason"]) for s in skipped])
print("slices:", names)

## 5. Test A: does the class text steer the mask? (pre-op-pair start, only the text changes)

In [ ]:
from rhuh_synthesis.overfit import (add_prompt_embeddings, class_set, generate, image_start_latents, mask_latents,
                                   prompt_caption, save_overfit_pages, score_slices, summarize, text_control_readings)
from rhuh_synthesis.text import scenario_prompts

A = TP["A_text_control"]
GENERATION = dict(num_inference_steps=config["sampling"]["num_inference_steps"],
                  guidance_scale=config["sampling"]["guidance_scale"], seed=S4["generation"]["seed"],
                  batch=S4["generation"]["batch_pairs"], mask_codec=config["representation"]["mask_codec"])
RUN.mkdir(parents=True)
(RUN / "arrays").mkdir()
started = time.perf_counter()
presence = A["presence_mm2"]
own_sets = [class_set(t["label"], pixel_mm=pixel_mm, presence_mm2=presence) for t in targets]
scenario_ids = {name: tuple(ids) for name, ids in A["scenarios"].items() if name != "own"}
pairs = {"own": [cache["patients"][p]["prompts"][k][0] for p, k in items]}
pairs.update({name: [scenario_prompts(ids, config["text"])["prompts"][0]] * len(items) for name, ids in scenario_ids.items()})
requested = {"own": own_sets, **{name: [ids] * len(items) for name, ids in scenario_ids.items()}}
record["added_prompts"] = add_prompt_embeddings(cache, [p for plist in pairs.values() for pair in plist for p in pair], encode)
pre_image = image_start_latents(cache, items, "pre")
pre_mask = mask_latents(vae, np.stack([exports[p]["pre_label"][k] for p, k in items]),
                        mask_codec=config["representation"]["mask_codec"], device="cuda")
test_a, generated = {}, {}
for name, scenario_pairs in pairs.items():
    images, masks = generate(unet11, vae, ddim, cache, items, start_image=pre_image, start_mask=pre_mask,
                             prompts=scenario_pairs, **GENERATION)
    generated[name] = images
    rows = score_slices(images, masks, targets, pixel_mm=pixel_mm, foreground_threshold=DIAG["metrics"]["foreground_threshold"])
    adherence = [{"name": n, "requested": list(req), "own": list(own),
                  "generated": list(class_set(m, pixel_mm=pixel_mm, presence_mm2=presence))}
                 for n, req, own, m in zip(names, requested[name], own_sets, masks)]
    panels = [{"name": f"{n} | requested {list(req) or 'none'} | generated {a['generated'] or 'none'}",
               "caption": prompt_caption(cache, item, pair), "pre_image": t["pre_image"], "post_image": t["post_image"],
               "generated_image": i, "label": t["label"], "generated_mask": m}
              for n, item, pair, req, a, t, i, m in zip(names, items, scenario_pairs, requested[name], adherence,
                                                         targets, images, masks)]
    save_overfit_pages(RUN / "grids" / f"A_{name}.png", panels, title=f"Text control {RUN_TAG} A {name}")
    np.savez_compressed(RUN / "arrays" / f"A_{name}.npz", images=images.astype(np.float16), masks=masks)
    test_a[name] = {"summary": summarize(rows, categories, groups=groups), "adherence": adherence,
                    "slices": [dict(row, name=n) for row, n in zip(rows, names)]}
    print(f"{name}: adherence {sum(a['generated'] == a['requested'] for a in adherence)}/16 |",
          {k: test_a[name]["summary"][k] for k in ("slice_mae_mean", "id2_dice_mean", "id3_overlapped", "label_free_labeled_mm2_mean")})
readings_a = text_control_readings({name: v["adherence"] for name, v in test_a.items() if name != "own"}, A["reading"])
readings_a["own_adherence"] = sum(a["generated"] == a["requested"] for a in test_a["own"]["adherence"])
readings_a["own_slice_mae"] = test_a["own"]["summary"]["slice_mae_mean"]
readings_a["reproduces_r4"] = abs(readings_a["own_slice_mae"] - A["r4_pair_own_preop_slice_mae"]) <= A["r4_tolerance"]
readings_a["image_change_vs_own"] = {name: float(np.abs(generated[name] - generated["own"]).mean())
                                     for name in generated if name != "own"}
record["test_a_seconds"] = time.perf_counter() - started
print(json.dumps(readings_a, indent=1))

## 6. Test B: what open VLMs say about the pre-op slices (local models only)

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

from rhuh_synthesis.caption_check import agreement, caption_flags, slice_to_rgb, token_count, vlm_caption

B = TP["B_caption_feasibility"]
del unet11
gc.collect()
torch.cuda.empty_cache()
started = time.perf_counter()
pre_images = [slice_to_rgb(exports[p]["pre_image"][k]) for p, k in items]
has_label = [bool((exports[p]["pre_label"][k] > 0).any()) for p, k in items]
template = config["text"]["image_template"]
candidates = [("general", B["models"]["general"])] + [("medical", repo) for repo in B["models"]["medical_candidates"]]
captions, model_status, medical_done = [], {}, False
for role, repo in candidates:
    if role == "medical" and medical_done:
        continue
    try:
        processor = AutoProcessor.from_pretrained(repo)
        vlm = AutoModelForImageTextToText.from_pretrained(repo, dtype=torch.bfloat16, device_map="cuda").eval()
    except Exception as exc:
        model_status[repo] = f"not loaded: {type(exc).__name__}: {str(exc)[:300]}"
        print(repo, model_status[repo])
        continue
    model_started = time.perf_counter()
    for style, prompt in B["styles"].items():
        for n, image, labelled in zip(names, pre_images, has_label):
            text = vlm_caption(vlm, processor, image, prompt, max_new_tokens=B["max_new_tokens"][style])
            captions.append({"model": repo, "role": role, "style": style, "name": n, "has_pre_label": labelled,
                             "caption": text, "clip_tokens": token_count(tokenizer, text),
                             "clip_tokens_in_template": token_count(tokenizer, template.format(organ=config["text"]["organ"], kind=text)),
                             **caption_flags(text)})
    model_status[repo] = f"ok in {time.perf_counter() - model_started:.0f}s"
    print(repo, model_status[repo])
    medical_done = medical_done or role == "medical"
    del vlm, processor
    gc.collect()
    torch.cuda.empty_cache()
summary_b = {}
for repo, style in sorted({(c["model"], c["style"]) for c in captions}):
    rows = [c for c in captions if c["model"] == repo and c["style"] == style]
    summary_b[f"{repo} | {style}"] = {
        "captions": len(rows), "clip_tokens_mean": float(np.mean([r["clip_tokens"] for r in rows])),
        "clip_tokens_max": max(r["clip_tokens"] for r in rows),
        "over_limit_in_template": sum(r["clip_tokens_in_template"] > B["token_limit"] for r in rows), **agreement(rows)}
record["test_b_seconds"] = time.perf_counter() - started
print(json.dumps(summary_b, indent=1))
for c in captions:
    flag = "ABN" if c["mentions_abnormality"] else ("NORMAL" if c["says_normal"] else "-")
    print(f"[{c['role']}/{c['style']}] {c['name']} label={'yes' if c['has_pre_label'] else 'no'} "
          f"tokens={c['clip_tokens']}/{c['clip_tokens_in_template']} flag={flag}: {c['caption']}")

## 7. Write-once record and the part to send

In [ ]:
from rhuh_synthesis.manifest import write_once_json

record.update(test_a=test_a, readings_a=readings_a,
              test_b={"model_status": model_status, "captions": captions, "summary": summary_b},
              total_seconds=time.perf_counter() - STARTED, peak_gib=torch.cuda.max_memory_allocated() / 2**30)
path, digest = write_once_json(RUN / "record.json", record)
print("saved", path, digest[:12])
shareable = {
    "commit": commit, "config_sha256": record["config_sha256"][:12], "gpu": record["gpu"],
    "weights_sha256": record["weights"]["sha256"][:12],
    "test_a": {"readings": readings_a, "summaries": {name: v["summary"] for name, v in test_a.items()},
               "adherence": {name: v["adherence"] for name, v in test_a.items()}},
    "test_b": {"model_status": model_status, "summary": summary_b},
    "seconds": {"test_a": round(record["test_a_seconds"]), "test_b": round(record["test_b_seconds"]),
                "total": round(record["total_seconds"])},
    "peak_gib": round(record["peak_gib"], 1),
}
print(json.dumps(shareable, indent=1))

## Next

Save this notebook with its outputs under a name Git ignores (for example `RHUH_GBM_Colab_text_control_captionsipynb.ipynb`). Then download `record.json` and the `grids/` pages from `synthesis/EXP-2026-001/text_control_captions/<RUN_TAG>/` on Drive into the same folder on your laptop. The full captions are in `record.json` and in the printout of section 6.